# Safeguard 6 — RWA data-quality acceptance

Reproducible, read-only validation of the candidate RWA catalog, registry, daily reconciliation, public payload boundaries, and pilot ledger. Snapshot date: **2026-07-30**.

## TL;DR

**Local acceptance passed.** The rebuilt matrix retains **5,161 of 5,161** source instruments; all **3,438** RWA.xyz token rows remain represented across **3,435** normalized contracts with zero cross-asset contract collisions. Semantic normalization reduced **55 raw mixed-class ids to zero canonical mixed-class ids**, while two bare-ticker ambiguities remain explicitly source-scoped. Only **93 of 1,169 RWA.xyz assets (7.96%)** are verified, so unverified mappings remain candidates. All 13 profiled default public endpoints returned HTTP 200 below **1,000,000 bytes** and **2.5 seconds** locally. The daily snapshot reconciles, but stale Hyperliquid and derivative catalog components keep promotion fail-closed. The schema-v3 pilot ledger accepted valid evidence and rejected both crossed-market claims and automatic promotion.

## Context and methods

The analysis uses repository artifacts and application builders only; it does not fetch external data or mutate production. Counts are evaluated at their declared grains: RWA.xyz source asset, RWA.xyz token listing, normalized network/address contract, canonical asset, and venue instrument. Response-size acceptance uses serialized HTTP response bytes. Latency is a cold-process local acceptance signal, not a production SLO.

In [1]:
from __future__ import annotations

from copy import deepcopy
from datetime import UTC, datetime, timedelta
import json
import math
import os
from pathlib import Path
import sys
from tempfile import TemporaryDirectory
from time import perf_counter

REPO = Path.cwd().resolve()
if REPO.name == 'analysis':
    REPO = REPO.parent
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from fastapi.testclient import TestClient  # noqa: E402 -- repo path is configured above
from scripts.run_rwa_growth_pilot import PILOT_FEEDS  # noqa: E402
from src.resource_server import app  # noqa: E402
from src.rwa_asset_identity import build_rwa_ticker_identity_audit  # noqa: E402
from src.rwa_coverage import build_rwa_asset_matrix  # noqa: E402
from src.rwa_daily_feed_agent import load_daily_feed_agent_report  # noqa: E402
from src.rwa_store import RWAObservationStore  # noqa: E402
from src.rwa_xyz_monitor import load_rwa_xyz_monitor_report  # noqa: E402

print({'repository': str(REPO), 'executed_at': datetime.now(UTC).isoformat()})

{'repository': '/Users/johannfocke/Documents/Antigravity/Agentic Payments', 'executed_at': '2026-07-30T16:00:53.672662+00:00'}


## Data

Load the authoritative current artifacts and rebuild the lossless matrix from source components.

In [2]:
matrix = build_rwa_asset_matrix()
xyz = load_rwa_xyz_monitor_report()
daily = load_daily_feed_agent_report()
identity_audit = build_rwa_ticker_identity_audit()

matrix_summary = matrix['summary']
xyz_summary = xyz['summary']
contract_quality = xyz_summary['contract_identity_quality']
identity_quality = xyz_summary['identity_quality']
freshness_manifest = matrix['source_snapshot_manifest']

loaded = {
    'canonical_assets': matrix_summary['canonical_asset_count'],
    'coverage_rows': matrix_summary['coverage_row_count'],
    'nested_instruments': matrix_summary['nested_instrument_count'],
    'rwa_xyz_assets': xyz_summary['asset_count'],
    'rwa_xyz_tokens': xyz_summary['token_count'],
    'unique_contracts': contract_quality['unique_contract_identity_count'],
}
print(json.dumps(loaded, indent=2))

{
  "canonical_assets": 2139,
  "coverage_rows": 5161,
  "nested_instruments": 5161,
  "rwa_xyz_assets": 1169,
  "rwa_xyz_tokens": 3438,
  "unique_contracts": 3435
}


## Results — completeness and identity

The nested instrument array is authoritative. Compatibility projections are not allowed to reduce the retained source-instrument count. Contract-level acceptance permits repeated source listings for the same RWA.xyz asset but blocks incomplete identities or a contract assigned to multiple source assets.

In [3]:
assert matrix_summary['coverage_row_count'] == matrix_summary['nested_instrument_count']
assert matrix_summary['coverage_row_count'] == freshness_manifest['included_coverage_row_count']
assert matrix['matrix_schema']['authoritative_venue_grain'] == 'venues.<venue_id>.instruments[]'
assert contract_quality['preserved_token_row_count'] == xyz_summary['token_count']
assert contract_quality['invalid_token_row_count'] == 0
assert contract_quality['cross_asset_contract_collision_group_count'] == 0
assert contract_quality['decision_grade_acceptance']['accepted'] is True

completeness_result = {
    'matrix_lossless': True,
    'coverage_rows': matrix_summary['coverage_row_count'],
    'nested_instruments': matrix_summary['nested_instrument_count'],
    'token_rows_preserved': contract_quality['preserved_token_row_count'],
    'unique_contracts': contract_quality['unique_contract_identity_count'],
    'benign_same_asset_duplicate_groups': contract_quality['benign_duplicate_source_listing_group_count'],
    'cross_asset_contract_collisions': contract_quality['cross_asset_contract_collision_group_count'],
    'contract_decision_gate': contract_quality['decision_grade_acceptance']['status'],
}
print(json.dumps(completeness_result, indent=2))

{
  "matrix_lossless": true,
  "coverage_rows": 5161,
  "nested_instruments": 5161,
  "token_rows_preserved": 3438,
  "unique_contracts": 3435,
  "benign_same_asset_duplicate_groups": 3,
  "cross_asset_contract_collisions": 0,
  "contract_decision_gate": "pass"
}


In [4]:
identity_summary = identity_audit['summary']
matrix_identity_quality = matrix_summary['identity_quality']
cross_class_unresolved = matrix_identity_quality['decision_grade_mixed_class_asset_id_count']
assert matrix_identity_quality['acceptance']['status'] == 'pass'
assert matrix_identity_quality['canonical_mixed_class_asset_id_count'] == 0
assert cross_class_unresolved == 0
assert identity_quality['decision_grade_mixed_class_asset_id_count'] == 0
assert identity_quality['verified_asset_count'] + identity_quality['unverified_asset_count'] == identity_quality['denominator_asset_count']

identity_result = {
    'raw_mixed_class_ids_detected': matrix_identity_quality['raw_mixed_class_asset_id_count'],
    'canonical_mixed_class_ids': matrix_identity_quality['canonical_mixed_class_asset_id_count'],
    'decision_grade_cross_class_unresolved': cross_class_unresolved,
    'ambiguous_source_scoped_assets': matrix_identity_quality['ambiguous_source_scoped_asset_count'],
    'rwa_xyz_mixed_class_asset_ids': identity_quality['decision_grade_mixed_class_asset_id_count'],
    'verified_assets': identity_quality['verified_asset_count'],
    'unverified_assets': identity_quality['unverified_asset_count'],
    'verified_rate': identity_quality['verified_asset_rate'],
    'boundary': 'Unverified source-scoped identities remain candidates and are not promoted as canonical underlyings.',
}
print(json.dumps(identity_result, indent=2))

{
  "raw_mixed_class_ids_detected": 55,
  "canonical_mixed_class_ids": 0,
  "decision_grade_cross_class_unresolved": 0,
  "ambiguous_source_scoped_assets": 2,
  "rwa_xyz_mixed_class_asset_ids": 0,
  "verified_assets": 93,
  "unverified_assets": 1076,
  "verified_rate": 0.079555,
  "boundary": "Unverified source-scoped identities remain candidates and are not promoted as canonical underlyings."
}


## Results — yield semantics

Normalized numerical values retain their raw trend objects and declare both unit and basis. Non-finite values are not accepted.

In [5]:
yield_checks = {}
for metric, value_field, raw_field, expected_unit in [
    ('yield_to_maturity', 'yield_to_maturity_value', 'yield_to_maturity_raw_trend', 'decimal_fraction'),
    ('apy_30_day', 'apy_30_day_value', 'apy_30_day_raw_trend', 'percentage_points'),
]:
    populated = [row for row in xyz['asset_rows'] if row.get(value_field) is not None]
    assert all(isinstance(row[value_field], (int, float)) and math.isfinite(row[value_field]) for row in populated)
    assert all(row.get(raw_field) is not None for row in populated)
    assert all(row.get(f'{metric}_unit') == expected_unit for row in populated)
    yield_checks[metric] = {
        'populated_assets': len(populated),
        'unit': expected_unit,
        'basis': xyz_summary['yield_metric_quality'][metric]['basis'],
        'raw_retained': True,
    }
print(json.dumps(yield_checks, indent=2))

{
  "yield_to_maturity": {
    "populated_assets": 24,
    "unit": "decimal_fraction",
    "basis": "annualized_yield_to_maturity",
    "raw_retained": true
  },
  "apy_30_day": {
    "populated_assets": 175,
    "unit": "percentage_points",
    "basis": "annual_percentage_yield_trailing_30_day",
    "raw_retained": true
  }
}


## Results — freshness and daily reconciliation

Assembly time is not represented as source freshness. Each dynamic component carries its own snapshot timestamp and cadence; stale components keep the aggregate matrix out of promotion-ready state. The daily agent binds comparisons to a canonical SHA-256 snapshot and creates an explicit baseline when no distinct prior snapshot exists.

In [6]:
dynamic_components = [
    row for row in freshness_manifest['components']
    if row['freshness_status'] != 'not_time_series_static_catalog'
]
stale_components = [row['component_id'] for row in dynamic_components if row['freshness_status'] == 'stale']
assert freshness_manifest['all_dynamic_components_current'] is False
assert stale_components
assert daily['status']['acceptance'] == 'passed'
assert daily['status']['decision_usable'] is True
assert daily['status']['snapshot_reconciled'] is True
assert daily['source_snapshot']['asset_count'] == xyz_summary['asset_count']
assert daily['source_snapshot']['token_count'] == xyz_summary['token_count']
assert daily['source_snapshot']['unique_token_contract_count'] == contract_quality['unique_contract_identity_count']
assert len(daily['source_snapshot']['canonical_json_sha256']) == 64

freshness_result = {
    'all_dynamic_components_current': freshness_manifest['all_dynamic_components_current'],
    'stale_components': stale_components,
    'promotion_ready': False,
    'daily_status': daily['status']['acceptance'],
    'daily_alert_level': daily['summary']['alert_level'],
    'daily_snapshot_sha256': daily['source_snapshot']['canonical_json_sha256'],
}
print(json.dumps(freshness_result, indent=2))

{
  "all_dynamic_components_current": false,
  "stale_components": [
    "hyperliquid_tradeable_discovery",
    "derivative_venue_discovery"
  ],
  "promotion_ready": false,
  "daily_status": "passed",
  "daily_alert_level": "baseline_created",
  "daily_snapshot_sha256": "f07f4bed75a032c53ac226ebd0fda531cb2a31457bf0479d624dad13cc593a42"
}


## Results — public endpoint boundaries

Default catalog responses must be useful summaries or bounded first pages, not multi-megabyte bulk exports. Completeness remains reachable through deterministic pagination and exact resolution, which is covered by the automated acceptance suite. The local cold-process latency ceiling below is deliberately conservative and is not presented as a production SLO.

In [7]:
MAX_DEFAULT_BYTES = 1_000_000
MAX_LOCAL_SECONDS = 2.5
PROFILE_ENDPOINTS = [
    '/v1/rwa/registry',
    '/v1/rwa/sourcing/jobs',
    '/v1/rwa/registry/venues',
    '/v1/rwa/discovery',
    '/v1/rwa/derivative-venues',
    '/v1/rwa/identity-audit',
    '/v1/rwa/non-crypto-feeds',
    '/v1/rwa/provider-catalog',
    '/v1/rwa/assets',
    '/v1/rwa/coverage',
    '/v1/rwa/consensus/sources',
    '/v1/rwa/market-expansion',
    '/v1/rwa/equity-universes',
]

client = TestClient(app)
endpoint_profile = []
for endpoint in PROFILE_ENDPOINTS:
    started = perf_counter()
    response = client.get(endpoint)
    elapsed = perf_counter() - started
    row = {
        'endpoint': endpoint,
        'status_code': response.status_code,
        'bytes': len(response.content),
        'seconds': round(elapsed, 4),
        'within_byte_budget': len(response.content) <= MAX_DEFAULT_BYTES,
        'within_latency_budget': elapsed <= MAX_LOCAL_SECONDS,
    }
    endpoint_profile.append(row)

assert all(row['status_code'] == 200 for row in endpoint_profile)
assert all(row['within_byte_budget'] for row in endpoint_profile)
assert all(row['within_latency_budget'] for row in endpoint_profile)
print(json.dumps(endpoint_profile, indent=2))

2026-07-30 11:00:57,053 [INFO] httpx: HTTP Request: GET http://testserver/v1/rwa/registry "HTTP/1.1 200 OK"


2026-07-30 11:00:58,323 [INFO] httpx: HTTP Request: GET http://testserver/v1/rwa/sourcing/jobs "HTTP/1.1 200 OK"


2026-07-30 11:00:59,218 [INFO] httpx: HTTP Request: GET http://testserver/v1/rwa/registry/venues "HTTP/1.1 200 OK"


2026-07-30 11:01:00,163 [INFO] httpx: HTTP Request: GET http://testserver/v1/rwa/discovery "HTTP/1.1 200 OK"


2026-07-30 11:01:00,521 [INFO] httpx: HTTP Request: GET http://testserver/v1/rwa/derivative-venues "HTTP/1.1 200 OK"


...

2026-07-30 11:01:03,858 [INFO] httpx: HTTP Request: GET http://testserver/v1/rwa/coverage "HTTP/1.1 200 OK"


2026-07-30 11:01:05,334 [INFO] httpx: HTTP Request: GET http://testserver/v1/rwa/consensus/sources "HTTP/1.1 200 OK"


2026-07-30 11:01:06,318 [INFO] httpx: HTTP Request: GET http://testserver/v1/rwa/market-expansion "HTTP/1.1 200 OK"


2026-07-30 11:01:07,268 [INFO] httpx: HTTP Request: GET http://testserver/v1/rwa/equity-universes "HTTP/1.1 200 OK"


[
  {
    "endpoint": "/v1/rwa/registry",
    "status_code": 200,
    "bytes": 293545,
    "seconds": 1.0138,
    "within_byte_budget": true,
    "within_latency_budget": true
  },
  {
    "endpoint": "/v1/rwa/sourcing/jobs",
    "status_code": 200,
    "bytes": 53023,
    "seconds": 1.2706,
    "within_byte_budget": true,
    "within_latency_budget": true
  },
  {
    "endpoint": "/v1/rwa/registry/venues",
    "status_code": 200,
    "bytes": 143749,
    "seconds": 0.8944,
    "within_byte_budget": true,
    "within_latency_budget": true
  },
  {
    "endpoint": "/v1/rwa/discovery",
    "status_code": 200,
    "bytes": 224844,
    "seconds": 0.9459,
    "within_byte_budget": true,
    "within_latency_budget": true
  },
  {
    "endpoint": "/v1/rwa/derivative-venues",
    "status_code": 200,
    "bytes": 101262,
    "seconds": 0.3575,
    "within_byte_budget": true,
    "within_latency_budget": true
  },
  {
    "endpoint": "/v1/rwa/identity-audit",
    "status_code": 200,
    "bytes":

## Results — pilot ledger evidence boundary

A temporary schema-v3 ledger proves three controls: valid raw evidence is stored, contradictory crossed-market claims are rejected, and no capture may self-promote to production.

In [8]:
def successful_capture(feed: dict, checked_at: datetime) -> dict:
    return {
        **feed,
        'started_at': (checked_at - timedelta(seconds=2)).isoformat(),
        'checked_at': checked_at.isoformat(),
        'status': 'ok',
        'checks': {
            'freshness_seconds': 1.0,
            'freshness_limit_seconds': feed['freshness_limit_seconds'],
            'freshness_pass': True,
            'bidask_sanity_pass': True,
        },
        'raw_observation': {
            'symbol': feed['symbol'],
            'venue': feed['venue'],
            'source_type': 'venue_api_order_book',
            'timestamp': checked_at.isoformat(),
            'bid': 99.0,
            'ask': 101.0,
        },
        'production_promoted': False,
    }

with TemporaryDirectory() as temp_dir:
    store = RWAObservationStore(str(Path(temp_dir) / 'rwa.db'))
    capture = successful_capture(PILOT_FEEDS[0], datetime.now(UTC) - timedelta(seconds=5))
    stored = store.store_pilot_outcomes([capture])[0]

    crossed = deepcopy(capture)
    crossed['checked_at'] = (datetime.now(UTC) - timedelta(seconds=3)).isoformat()
    crossed['started_at'] = (datetime.now(UTC) - timedelta(seconds=4)).isoformat()
    crossed['raw_observation']['timestamp'] = crossed['checked_at']
    crossed['raw_observation'].update({'bid': 200.0, 'ask': 100.0})
    crossed['checks']['bidask_sanity_pass'] = True
    crossed_rejected = False
    try:
        store.store_pilot_outcomes([crossed])
    except ValueError:
        crossed_rejected = True

    promoted = deepcopy(capture)
    promoted['checked_at'] = (datetime.now(UTC) - timedelta(seconds=1)).isoformat()
    promoted['started_at'] = (datetime.now(UTC) - timedelta(seconds=2)).isoformat()
    promoted['raw_observation']['timestamp'] = promoted['checked_at']
    promoted['production_promoted'] = True
    promotion_rejected = False
    try:
        store.store_pilot_outcomes([promoted])
    except ValueError:
        promotion_rejected = True

    ledger_rows = store.list_pilot_outcomes(include_evidence=True)
    schema_status = store.schema_status()

assert stored['inserted'] is True
assert crossed_rejected is True
assert promotion_rejected is True
assert len(ledger_rows) == 1
assert schema_status['ready'] is True and schema_status['schema_version'] == 3
ledger_result = {
    'schema_version': schema_status['schema_version'],
    'valid_evidence_inserted': stored['inserted'],
    'crossed_claim_rejected': crossed_rejected,
    'automatic_promotion_rejected': promotion_rejected,
    'retained_rows': len(ledger_rows),
}
print(json.dumps(ledger_result, indent=2))

{
  "schema_version": 3,
  "valid_evidence_inserted": true,
  "crossed_claim_rejected": true,
  "automatic_promotion_rejected": true,
  "retained_rows": 1
}


## Takeaways

Safeguard 6 can pass locally only when every assertion above completes. Passing establishes lossless catalog assembly, explicit identity uncertainty, contract collision controls, bounded default responses, deterministic daily reconciliation, and a fail-closed evidence ledger. It does **not** authorize production promotion: stale derivative and Hyperliquid catalog snapshots, the required 14-day pilot window, source rights, benchmark alignment, manipulation/depth review, source independence, and human approval remain separate release gates.